In [76]:
import pandas as pd 

df = pd.read_excel("raw_agexport_db.xlsx")

In [77]:
import ast 

for col in df.columns:
    if col != "partner_name":
        df[col] = df[col].apply(ast.literal_eval)

In [78]:
display(df)

,partner_name,available_services,phone_number,emergency_number,location
0,Centro Dental San Lucas,"[Odontología de adultos, odontogeriatria, Odon...","[40635851, 40053818]",[40635851],"[Km. 29.9, Carr. Interamericana Plaza San Luca..."
1,GrupoDent,"[Endodoncia, Odontología preventiva, Ortodonci...",[22699800],[],"[6a. Av. 6-63, Zona 10, Planta Baja, Local 4]"
2,Facultad Odontología UFM,"[Odontología, servicios dentales, Emergencias ...",[23387850],[],[6a. Calle Manuel Ayau 7-11 zona 10]
3,Clidenth,"[Servicios dentales odontológicos, odontología...",[77647496],[],"[6ta Avenida 9-54, Colonia El Bosque, Edificio..."
4,Centro Dental Kyrios,"[Endodoncia, Odontología preventiva, Ortodonci...",[24255657],[42794922],"[C.C. Tikal Futura, Torre Sol, Nivel 10, Zona..."
5,Ríe Genial,"[Odontología preventiva, Odontología restaurat...","[22286665, 24380262]",[30679896],[2 Calle 36-61 zona 7 Condominio Arboleda Casa...
6,Vitalmed Odontología,"[Odontología, servicios dentales, Emergencias ]",[23399210],[38268176],"[8 CALLE PONIENTE 80 ZONA 0 ANTIGUA-00, ZONA 0]"
7,Clínicas Sonríe,[Servicios dentales ],[24447878],[24447878],"[Diagonal 6, 13-01 Z.10 Planta Baja, Nueva Fas..."
8,Dental Core,"[Servicios dentales odontológicos, odontología...",[41835895],[41835895],[GEMINIS 10]
9,G SMILE,"[Servicios dentales odontológicos, odontología...",[55270557],[55270557],"[20 AVENIDA 2-22 ZONA 3, EDIFICIO COVADONGA OF..."


In [79]:
# Create all combinations
df_exploded = df.explode("location")

addresses = []
# Print in desired format
for _, row in df_exploded.iterrows():
    addresses.append(f"{row['partner_name']} - {row['location']}")

In [81]:
from typing import Optional, Dict, Any
import googlemaps

def lookup_place_fields(
    gmaps: googlemaps.Client,
    query: str,
    *,
    bias_location: Optional[tuple[float, float]] = None,  # (lat, lon)
    radius_m: int = 25000,
) -> Optional[Dict[str, Any]]:
    """
    Resolve a textual query/address to a specific Google Place and return:
    name, address, lat, lon, place_id, maps_url

    Requires Google Places API enabled on your key/project.

    Parameters
    ----------
    gmaps : googlemaps.Client
        googlemaps client initialized with your API key.
    query : str
        Free-form text like "CVS Pharmacy 5th Ave San Diego CA" or a full address.
    bias_location : (lat, lon), optional
        If you know roughly where the place is, provide a location bias to improve results.
    radius_m : int
        Radius for location bias.

    Returns
    -------
    dict or None
        {
          "name": str,
          "address": str,
          "lat": float,
          "lon": float,
          "place_id": str,
          "maps_url": str
        }
        or None if no match found.
    """
    fields = ["place_id", "name", "formatted_address", "geometry"]

    find_kwargs = {
        "input": query,
        "input_type": "textquery",
        "fields": fields,
    }

    # Optional location bias to disambiguate chains with many nearby locations
    if bias_location:
        find_kwargs["location_bias"] = f"circle:{radius_m}@{bias_location[0]},{bias_location[1]}"

    resp = gmaps.find_place(**find_kwargs)

    candidates = resp.get("candidates") or []
    if not candidates:
        return None

    c = candidates[0]

    place_id = c.get("place_id")
    name = c.get("name")
    address = c.get("formatted_address")

    lat = lon = None
    geom = (c.get("geometry") or {}).get("location") or {}
    if "lat" in geom and "lng" in geom:
        lat = float(geom["lat"])
        lon = float(geom["lng"])

    # If geometry wasn't returned for some reason, fetch it via Place Details
    if place_id and (lat is None or lon is None or not address or not name):
        details = gmaps.place(
            place_id=place_id,
            fields=["name", "formatted_address", "geometry"]
        ).get("result") or {}

        name = name or details.get("name")
        address = address or details.get("formatted_address")

        dloc = (details.get("geometry") or {}).get("location") or {}
        if lat is None and "lat" in dloc:
            lat = float(dloc["lat"])
        if lon is None and "lng" in dloc:
            lon = float(dloc["lng"])

    if not place_id:
        return None

    maps_url = f"https://www.google.com/maps/place/?q=place_id:{place_id}"

    return {
        "query": query, 
        "name": name,
        "address": address,
        "lat": lat,
        "lon": lon,
        "place_id": place_id,
        "maps_url": maps_url,
    }


In [ ]:
import googlemaps

gmaps = googlemaps.Client(key=GOOGLE_API_KEY)

In [87]:
results = []
for address in addresses: 
    row = lookup_place_fields(gmaps, address)
    if row: 
        results.append(row)
    else: 
        results.append({
            "query": address, 
            "name": None,
            "address": None,
            "lat": None,
            "lon": None,
            "place_id": None,
            "maps_url": None,
        })

In [88]:
# cleaned_result = [x for x in results if x is not None]
cleaned_df = pd.DataFrame(results)

In [89]:
cleaned_df.to_excel("agexport_googlereference.xlsx")

In [85]:
cleaned_df

,query,name,address,lat,lon,place_id,maps_url
0,"Centro Dental San Lucas - Km. 29.9, Carr. Inte...",Centro Dental San Lucas,"Pan-American Highway 9, San Lucas Sacatepéquez...",14.572324,-90.482318,ChIJq0XgwXujiYUR-4JnHIl1dD8,https://www.google.com/maps/place/?q=place_id:...
1,"GrupoDent - 6a. Av. 6-63, Zona 10, Planta Baja...",GrupoDent Sixtino,"planta baja, Sixtino I, 6A Avenida 6-63, Cdad....",14.606977,-90.509703,ChIJ0_nOycujiYURmcpW4O4kbTQ,https://www.google.com/maps/place/?q=place_id:...
2,Facultad Odontología UFM - 6a. Calle Manuel Ay...,Facultad de Odontologia UFM,"6A Calle 7-11, Cdad. de Guatemala 01010, Guate...",14.607767,-90.509044,ChIJ3xIHTsqjiYUR4aSehFxTlUY,https://www.google.com/maps/place/?q=place_id:...
3,"Clidenth - 6ta Avenida 9-54, Colonia El Bosque...",CLIDENTH,"8G6C+Q4W, Calz. Kaibil Balam, Huehuetenango, G...",15.311997,-91.479556,ChIJSSkhCCsUjIURP0CXZTcymGo,https://www.google.com/maps/place/?q=place_id:...
4,"Centro Dental Kyrios - C.C. Tikal Futura, Tor...",Centro Dental Kyrios,"22-43, Centro Comercial Tikal Futura Torre Sol...",14.623383,-90.553311,ChIJU1VKMqWhiYUR9EQzyR3PFhQ,https://www.google.com/maps/place/?q=place_id:...
...,...,...,...,...,...,...,...
70,San Gregorio Hotel y Spa - Km. 29.5 Carretera ...,San Gregorio Hotel & Spa,"KM 30 Carretera a Santa Elena Barillas,Lote #5...",14.444136,-90.500481,ChIJ24bKzROviYURkYjcPWffriE,https://www.google.com/maps/place/?q=place_id:...
71,"IBIOMED - Sixtino I, 6ta avenida 6-63 zona 10",Edificio Sixtino 1,"6A Avenida 6-63, Cdad. de Guatemala 01010, Gua...",14.606833,-90.510003,ChIJL2QKv8ujiYURzMX3cTBojk4,https://www.google.com/maps/place/?q=place_id:...
72,"INTERAMERICAN CAR RENTAL - 1a. Calle 5-20, Zon...",InterAmerican Car Rental,"1A Calle 5-20, Cdad. de Guatemala 01013, Guate...",14.603046,-90.529406,ChIJv6tJvl-hiYURFx2bZACewnA,https://www.google.com/maps/place/?q=place_id:...
73,Cedilab - 6 ave 9-18 zona 10 edificio Sixtino ...,Cedilab,"edificio Sixtino II, 6 avenida 9-18 zona 10, C...",14.603188,-90.510568,ChIJ7Yu30EKjiYURGtw6imznG74,https://www.google.com/maps/place/?q=place_id:...
